In [1]:
"""
get_adj_matrix.py
Author: Roberto Herrera

For each university we need 4 files:
    - {school}_Pubs.csv
    - {school}_plus_USDA_Pubs.csv
    - {school}_Grants.xlsx
    - {school}_fuzzy_names.json
Otherwise the code will skip that school.
"""

import os, json, math, re, traceback
import pandas as pd
import argparse
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



In [2]:
# -------------------- helpers --------------------
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    
def read_csv(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

# Shorten printed paths
def short(p, s=2):
    if not p:
        return "None"
    return "/".join(p.replace("\\", "/").split("/")[-s:])

# normalize authors name to match "Last, First Middle"
def normalize(name):
    name = str(name).strip()
    parts = re.split(r"\s+", name)
    if len(parts) >= 2:
        last = parts[-1]
        first = " ".join(parts[:-1])
        return f"{last}, {first}"
    return name

# scale node size for graph visualization
def scale_size(raw):
    MAX = 100
    return math.sqrt(raw / MAX) * 40 + 5

root = '..' + os.sep + 'LandGrantData' + os.sep

In [3]:
school = 'MU'

# pull necessary file paths
school_dir = os.path.join(root, school)
pubs_path  = os.path.join(school_dir, f"{school}_Pubs.csv")
plus_path  = os.path.join(school_dir, f"{school}_plus_USDA_Pubs.csv")
grants_files = [f for f in os.listdir(school_dir) if "Grant" in f and f.lower().endswith((".xlsx", ".csv"))]
grants_path = os.path.join(school_dir, grants_files[0]) if grants_files else None
json_path = os.path.join(school_dir, f"{school}_fuzzy_names.json")

print(f"Pubs file:   {short(pubs_path)}")
print(f"Plus file:   {short(plus_path)}")
print(f"Grants file: {short(grants_path)}")
print(f"Names JSON:  {short(json_path)}\n")

Pubs file:   MU/MU_Pubs.csv
Plus file:   MU/MU_plus_USDA_Pubs.csv
Grants file: MU/MUGrants15-26.xlsx
Names JSON:  MU/MU_fuzzy_names.json



In [4]:
# load faculty fuzzy-name mapping from JSON file
fuzzy_names = load_json(json_path)
faculty_list = list(fuzzy_names.keys())
fuzzy_map = {v.strip(): name for name, variants in fuzzy_names.items() for v in variants}

print(f"Total faculty in IPG: {len(faculty_list)}\n")

# ----------- Publication data -------------------------
# read and merge publication + USDA+ publication data
df_pubs = read_csv(pubs_path)
df_plus = read_csv(plus_path)
df = pd.concat([df_pubs, df_plus], ignore_index=True)#.set_index('Publication ID')
print(f"Total combined publications: {len(df)}\n")

Total faculty in IPG: 88

Total combined publications: 42518



In [113]:
# Of all the publications, remove those with no corresponding author
# Then take the subset of publications where USDA appears as an affiliation for corresponding authors

usda_institutes = ['United States Department of Agriculture', 'Agricultural Research Service', 'Biological Control of Insects Research']
data = df[~pd.isna(df['Corresponding Authors'])]
mask = pd.Series(False, data.index)
for inst in usda_institutes:
    mask |= data['Corresponding Authors'].str.contains(inst)
data = data[mask]

usda_folks = set()

for corr in data['Corresponding Authors']:
    # We still have to make sure that an author has a double affiliation
    # As opposed to having two separate corresponding authors
    
    aas = [a.strip() for a in corr.split("); ") if a.strip()]
    aas = pd.DataFrame([ aa.split(' (', maxsplit=1) for aa in aas ], columns=['Author','Affiliation']).set_index('Author')
    aas = aas.loc[~(aas['Affiliation'].isnull() | aas['Affiliation'].isna() )]
    aas['USDA'] = [ any([inst in affiliation for inst in usda_institutes]) for affiliation in aas['Affiliation'] ]

    # Fuzzy map to account for various author spellings
    # And to only consider IPG folks
    usda_folks |= {fuzzy_map[a] for a in aas[aas['USDA']].index if a in fuzzy_map}

In [114]:
usda_folks

{'Beissinger, Timothy M',
 'Best, Norman B',
 'Bilyeu, Kristin D',
 'Das, Debatosh',
 'FlintGarcia, Sherry A',
 'Gassmann, Walter',
 'Gillman, Jason D',
 'Hibbard, Bruce E',
 'Krishnan, Hari B',
 'Oliver, Melvin J',
 'Pereira, Adriano E',
 'Shelby, Kent S',
 'Shergill, Lovreet S',
 'Washburn, Jacob D'}

In [115]:
data[

,Publication ID,Title,Abstract,Source title,ISSN,Publisher,MeSH terms,PubYear,Open Access,Publication Type,...,Funder,Funder Group,Funder Country,Times cited,RCR,FCR,Altmetric,Fields of Research (ANZSRC 2020),Units of Assessment,Sustainable Development Goals
352,pub.1192596526,Genomic regions and candidate genes associated...,"Nitrogen (N), phosphorus (P), and sulfur (S) a...",PLOS ONE,1932-6203,Public Library of Science (PLoS),phosphorus; glycine max; seeds; nitrogen; sulf...,2025,All OA; Gold,Article,...,United States Department of Agriculture; Agric...,US Federal Funders; USDA - United States Depar...,United States; United States,0,NaN,NaN,NaN,"30 Agricultural, Veterinary and Food Sciences;...","A06 Agriculture, Veterinary and Food Science",NaN
727,pub.1191371216,Mesopolyploidy as a taxonomic clade marker for...,BACKGROUND AND AIMS: Whole Genome Duplications...,Annals of Botany,"0305-7364, 1095-8290",Oxford University Press (OUP),NaN,2025,Closed,Article,...,Directorate for Biological Sciences; Office of...,US Federal Funders; NSF - National Science Fou...,United States; United States; United States; U...,1,NaN,NaN,9.0,31 Biological Sciences; 3104 Evolutionary Biol...,A05 Biological Sciences,NaN
921,pub.1191030786,Seed-Specific Silencing of Abundantly Expresse...,Soybean meal (SBM) is extensively used as a pr...,International Journal of Molecular Sciences,"1661-6596, 1422-0067",MDPI,"glycine max; seeds; trypsin inhibitor, bowman-...",2025,All OA; Gold,Article,...,Agricultural Research Service,US Federal Funders; USDA - United States Depar...,United States,0,NaN,NaN,NaN,31 Biological Sciences; 3101 Biochemistry and ...,"A06 Agriculture, Veterinary and Food Science",NaN
1046,pub.1190619608,Why do some predicted protein structures fold ...,Abstract Protein structure prediction tools h...,bioRxiv,2692-8205,Cold Spring Harbor Laboratory,NaN,2025,All OA; Green,Preprint,...,Agricultural Research Service; United States D...,US Federal Funders; USDA - United States Depar...,United States; United States; United States,0,NaN,NaN,19.0,31 Biological Sciences; 3101 Biochemistry and ...,A05 Biological Sciences,NaN
1355,pub.1189561420,In situ growth of carbon nanotubes in biomass ...,Carbon nanotubes (CNTs) and their allotropes h...,Applied Physics Letters,"0003-6951, 1077-3118",AIP Publishing,NaN,2025,Closed,Article,...,US Forest Service,US Federal Funders; USDA - United States Depar...,United States,0,NaN,NaN,NaN,40 Engineering,B12 Engineering,7 Affordable and Clean Energy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42503,pub.1160561812,Genetic evidence that brassinosteroids suppres...,The developmental genetics of reproductive str...,Plant Direct,2475-4455,Wiley,NaN,2023,All OA; Gold,Article,...,Agricultural Research Service; National Instit...,US Federal Funders; USDA - United States Depar...,United States; United States; United States,8,0.84,3.19,6.0,31 Biological Sciences; 3105 Genetics; 3108 Pl...,A05 Biological Sciences,NaN
42505,pub.1166533194,Synthetically Labeled Images for Maize Plant D...,The detection of individual plants within fiel...,Lecture Notes in Computer Science,"0302-9743, 1611-3349",Springer Nature,NaN,2023,Closed,Chapter,...,Agricultural Research Service; United States D...,US Federal Funders; USDA - United States Depar...,United States; United States,0,NaN,0.00,NaN,46 Information and Computing Sciences; 4605 Da...,B11 Computer Science and Informatics,NaN
42506,pub.1170373738,Conservation and diversification of genes regu...,Brassinosteroids (BRs) are important regulator...,bioRxiv,2692-8205,Cold Spring Harbor Laboratory,NaN,2024,All OA; Green,Preprint,...,National Institute of Food and Agriculture; Ag...,US Federal Funders; USDA - United States Depar...,United States; United States,0,NaN,NaN,3.0,"30 Agricultural, Veterinary and Food Sciences;...","A06 Agriculture, Veterinary and Food Science",NaN
42508,pub.1172200195,Brassinosteroid biosynthesis and signaling: Co...,Brass